# Caso F · 01 MLflow + lakeFS — visión general

> _Tutorial · Caso de uso: **F — MLOps** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Entender los conceptos de experiment, run, artefacto y tag de dataset, y ver cómo MLflow + lakeFS resuelven la reproducibilidad sin reinventar la rueda.


## 2. Qué se aprende

- Diferencia entre experiment, run y artefacto.
- Cómo lakeFS versiona datasets como Git versiona código.
- Convención `experiment-name = caso-modelo-fecha`.


## 3. Contexto del caso de uso

El equipo F es transversal: define la convención que usarán todos los demás para que en semana 4 se pueda reproducir cualquier resultado.


## 4. Relación con CENTINELA+

MLflow va a producción cuando se desplieguen modelos reales. Versionar datasets evita el clásico 'funcionaba en mi máquina'.


## 5. Relación con Medallion

MLOps es **transversal**: versiona artefactos derivados de plata y oro.


## 6. Datos de entrada

Conceptual.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Construimos un mapa conceptual.


In [2]:
mlflow_concepts = pd.DataFrame(
    [
        ("Experiment", "Agrupación lógica", "case_B_baseline_2026"),
        ("Run", "Ejecución concreta", "rf_v3_seed42"),
        ("Param", "Hiperparámetro", "n_estimators=200"),
        ("Metric", "Indicador", "MAE=12.4"),
        ("Artifact", "Modelo / plot / dataset", "model.pkl, residuos.png"),
        ("Tag", "Metadata libre", "stage=staging"),
    ],
    columns=["objeto", "papel", "ejemplo"],
)
mlflow_concepts


,objeto,papel,ejemplo
0,Experiment,Agrupación lógica,case_B_baseline_2026
1,Run,Ejecución concreta,rf_v3_seed42
2,Param,Hiperparámetro,n_estimators=200
3,Metric,Indicador,MAE=12.4
4,Artifact,Modelo / plot / dataset,"model.pkl, residuos.png"
5,Tag,Metadata libre,stage=staging


## 10. Exploración paso a paso

lakeFS funciona como Git pero para datos: branches, commits y tags.


In [3]:
lakefs_concepts = pd.DataFrame(
    [
        ("Repository", "Bucket lógico (S3 backend)", "captia-datasets"),
        ("Branch", "Línea de trabajo", "main / experiment-Y"),
        ("Commit", "Snapshot inmutable", "deadbeef"),
        ("Tag", "Etiqueta humana", "case_B/baseline_v1"),
        ("Hooks", "Validaciones pre/post commit", "schema check"),
    ],
    columns=["objeto", "papel", "ejemplo"],
)
lakefs_concepts


,objeto,papel,ejemplo
0,Repository,Bucket lógico (S3 backend),captia-datasets
1,Branch,Línea de trabajo,main / experiment-Y
2,Commit,Snapshot inmutable,deadbeef
3,Tag,Etiqueta humana,case_B/baseline_v1
4,Hooks,Validaciones pre/post commit,schema check


## 11. Transformación bronce → plata

**Hello-world MLflow**: levantamos tracking local SQLite-style sobre filesystem y validamos la convención de naming antes de tocar datos.


In [4]:
import re
import mlflow

# Tracking URI local — equivalente a `sqlite:///mlflow.db` para esta clase
mlflow_dir = ROOT / "output" / "mlruns_overview"
mlflow_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(mlflow_dir.as_uri())

# Convención de naming canónica del equipo CAPTIA
EXPERIMENT_RE = re.compile(r"^case_[A-J]_(baseline|prod)_\d{4}$")
RUN_RE = re.compile(r"^[a-z]+_v\d+_seed\d+$")

def validate_experiment_name(name: str) -> bool:
    return bool(EXPERIMENT_RE.fullmatch(name))

def validate_run_name(name: str) -> bool:
    return bool(RUN_RE.fullmatch(name))

# Test de la convención
test_cases = [
    ("case_B_baseline_2026", True),
    ("case_C_prod_2026", True),
    ("case_X_test", False),  # X fuera del rango [A-J], y "test" fuera del set
    ("rf_v3_seed42", "run-name"),
]
for name, expected in test_cases[:3]:
    assert validate_experiment_name(name) == expected, f"Convencion fallo para {name}"
assert validate_run_name("rf_v3_seed42")
print("Convencion de naming OK ·",
      "experiment_name = '^case_[A-J]_(baseline|prod)_\\d{4}$'",
      "· run_name = '^[a-z]+_v\\d+_seed\\d+$'")


Convencion de naming OK · experiment_name = '^case_[A-J]_(baseline|prod)_\d{4}$' · run_name = '^[a-z]+_v\d+_seed\d+$'


## 12. Construcción de capa oro

Un primer experiment + run dummy para que el alumno **vea** la jerarquía completa y luego pueda hacer `mlflow ui` y navegar.


In [5]:
exp_name = "case_F_baseline_2026"
mlflow.set_experiment(exp_name)

with mlflow.start_run(run_name="hello_v1_seed42"):
    mlflow.log_params({"variant": "hello-world", "seed": SEED, "stage": "demo"})
    mlflow.log_metric("dummy_metric", 0.42)
    mlflow.set_tag("lakefs_tag", "case_F/demo_v1")
    mlflow.set_tag("dataset_uri", "lakefs://repo/main/HEAD")
    run_info = mlflow.active_run().info

print(f"Run creado: experiment={run_info.experiment_id}, run_id={run_info.run_id[:8]}...")
print(f"Para ver UI: mlflow ui --backend-store-uri {mlflow_dir.as_uri()}")

runs = mlflow.search_runs(experiment_names=[exp_name])
print(f"\nRuns en {exp_name}: {len(runs)}")
print(runs[["tags.mlflow.runName", "metrics.dummy_metric", "tags.lakefs_tag"]].to_string(index=False))


C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Run creado: experiment=376358567856422088, run_id=f4942525...
Para ver UI: mlflow ui --backend-store-uri file:///C:/CAPTIA/CAPTIA-SYNTHETIC-DATA-BMS/output/mlruns_overview

Runs en case_F_baseline_2026: 5
tags.mlflow.runName  metrics.dummy_metric tags.lakefs_tag
    hello_v1_seed42                  0.42  case_F/demo_v1
    hello_v1_seed42                  0.42  case_F/demo_v1
    hello_v1_seed42                  0.42  case_F/demo_v1
    hello_v1_seed42                  0.42  case_F/demo_v1
    hello_v1_seed42                  0.42  case_F/demo_v1


## 13. Visualizaciones explicativas

Diagrama relacional MLflow ↔ lakeFS + tabla de objetos invocados.


In [6]:
import matplotlib.pyplot as plt

# Tabla de objetos creados en este notebook (comprobables en mlflow ui)
objects_created = pd.DataFrame([
    {"objeto": "Experiment", "valor": exp_name, "ubicacion": "mlruns_overview/0/"},
    {"objeto": "Run", "valor": run_info.run_id[:12] + "...", "ubicacion": f"mlruns_overview/.../{run_info.run_id[:8]}..."},
    {"objeto": "Param", "valor": "variant=hello-world, seed=42", "ubicacion": "params/"},
    {"objeto": "Metric", "valor": "dummy_metric=0.42", "ubicacion": "metrics/"},
    {"objeto": "Tag", "valor": "lakefs_tag=case_F/demo_v1", "ubicacion": "tags/"},
])
print(objects_created.to_string(index=False))


    objeto                        valor                       ubicacion
Experiment         case_F_baseline_2026              mlruns_overview/0/
       Run              f49425251e95... mlruns_overview/.../f4942525...
     Param variant=hello-world, seed=42                         params/
    Metric            dummy_metric=0.42                        metrics/
       Tag    lakefs_tag=case_F/demo_v1                           tags/


## 14. Validaciones

Verificar que el run quedó persistido (idempotente: re-ejecutar añade otro run pero nunca rompe).


In [7]:
runs = mlflow.search_runs(experiment_names=[exp_name])
assert len(runs) >= 1, "No se persistió ningun run"
assert "tags.lakefs_tag" in runs.columns, "lakefs_tag tag no presente"
print(f"Validaciones OK · {len(runs)} run(s) en experiment '{exp_name}'")


Validaciones OK · 5 run(s) en experiment 'case_F_baseline_2026'


## 15. Errores comunes

1. Crear un run nuevo cada vez que cambias un hiperparámetro pero no marcar la experiment.
2. No registrar el lakeFS tag en el run.
3. Subir artefactos enormes (CSVs) en lugar de versionarlos en lakeFS.


## 16. Ejercicios propuestos

1. Diseña la convención de naming para tu equipo.
2. Escribe la regla `pre_commit` que valida el schema CAPTIA en lakeFS.
3. Discute cuándo subir un artefacto vs cuándo versionar el dataset.


## 17. Cómo se reutiliza con datos reales

El stack MLflow + lakeFS escala a producción sin cambios.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `06_case_F_mlops/02_tracking_experimentos.ipynb`.
- Documento web del caso: `docs/use-cases/case-f-mlops.md`.


## 19. Marco teórico (nivel doctoral)

### Versionado de datasets (lakeFS)

Modelo Git-like sobre object storage con commits inmutables:

$$
\text{commit}_t = \langle \text{tree}_t, \text{parent}_{t-1}, \text{metadata}_t \rangle
$$

con `tree` Merkle DAG. Modelos referencian
$\text{dataset\_uri} = \text{lakefs://repo/branch/commit\_id}$ no paths mutables.

### MLflow Run Schema

$$
\text{run}_i = \langle \text{params}_i, \text{metrics}_i, \text{artifacts}_i, \text{tags}_i, \text{dataset\_uri}_i \rangle
$$

### Reproducibilidad determinista

$$
\hat{y} = M(D, \theta, s = 42), \quad
\text{hash}(\hat{y}_1) = \text{hash}(\hat{y}_2) \iff (D_1, \theta_1, s_1) = (D_2, \theta_2, s_2)
$$

(ADR-008). Verificable con `pytest -m snapshot`.

### Linaje de features

$$
\text{Feature}_i = f_i(\text{Feature}_j, \text{Feature}_k, ...) \implies \text{DAG}_{features}
$$

Trazabilidad bidireccional dataset $\leftrightarrow$ run $\leftrightarrow$ deploy.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

MLOps no genera ROI directo, pero **reduce el coste de toda la cadena**. Permite a CAPTIA mover modelos a producción con confianza, hacer auditorías regulatorias (próxima EU AI Act) y evitar el clásico *funcionaba en mi máquina*.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción tiempo auditoría modelos (8 h → 1 h) | +800 €/año |
| Eliminación re-runs por incertidumbre | +400 €/año compute |
| Cumplimiento EU AI Act (riesgo evitado) | +20 000 € (riesgo) |
| **Bruto** | **+1 200 €/año** + risk hedging |


## 21. Bibliografía y referencias

- Zaharia, M. et al. (2018). *Accelerating the Machine Learning Lifecycle with MLflow*. CIDR.
- lakeFS Project. *Documentation*. https://docs.lakefs.io
- Sculley, D. et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.
- EU (2024). *Artificial Intelligence Act*. Regulation (EU) 2024/1689.


## 22. Etapa del pipeline · MLflow hello-world + naming convention CAPTIA

Convención: `experiment_name = ^case_[A-J]_(baseline|prod)_\d{4}$`. Sin convención, dos data scientists del IES Simarro registran `exp_test_2026` y `experimento_juan` — auditoría imposible al cabo de 3 meses.

> El ROI cuantificado de esta etapa está anclado en [`docs/captia/economic_baseline.md`](../../docs/captia/economic_baseline.md) — cualquier cifra de la sección 20 es derivable de ahí, no inventada.